# Interval Reentry Simulation Notebook
This notebook runs **interval-valued 3D atmospheric reentry simulations**, logs all major state intervals, and visualizes uncertainty growth.

Design goals:
- Strict interval safety (division-by-zero guarded)
- Clean, modular structure
- Append-only CSV logging
- Separate heat data logging
- Multi-run support with unique run IDs


## Imports and Configuration

In [5]:
import os
import csv
import math
import uuid
import matplotlib.pyplot as plt

from interval_math import Interval, interval_euler_step
import constants
import AtmosphereModel
from math_3d import f_interval, make_sigma_fn, aero_forces
from constants import HeatShield

from control import (
    SigmaControlStack,
    SigmaControlConfig,
    GuidanceScheduler,
    SimpleBankGuidance,
    BasicObservationProvider,
    BankActuator,
    BankActuatorLimits,
    ReentryState,
)


===== Interval Aero Forces Test =====



# Vehicle param + control stack setup

In [6]:
# ---------------- VEHICLE + TARGET ----------------
params = {
    "m": 5000.0,
    "S": 10.0,
    "CL": 0.3,
    "CD": 1.0,
}

target_phi = 0.3
target_lam = 1.0

# ---------------- CONTROL STACK ----------------
cfg = SigmaControlConfig(
    guidance_period_s=1.0,
    aero_enable_threshold_mps2=1.52,
)

scheduler = GuidanceScheduler(period_s=cfg.guidance_period_s)
guidance = SimpleBankGuidance()
obs_provider = BasicObservationProvider(target_phi, target_lam, params)

limits = BankActuatorLimits(
    sigma_rate_max_rps=math.radians(20.0),
    sigma_accel_max_rps2=math.radians(10.0),
)

bank_actuator = BankActuator(limits)

control = SigmaControlStack(
    cfg=cfg,
    scheduler=scheduler,
    guidance=guidance,
    obs_provider=obs_provider,
    bank_actuator=bank_actuator,
    rcs=None,
)

control.reset()

# ---------------- SIGMA FUNCTION ----------------
sigma_fn = make_sigma_fn(
    control=control,
    aero_force_fn=aero_forces,
    params=params,
)

## User Controls

In [ ]:

NUM_RUNS = 1        # change this line only
DT = 0.25 # 4 steps = 1 second
MAX_STEPS = 2000 # 500 seconds

DATA_DIR = "data_intv"
STATE_CSV = os.path.join(DATA_DIR, "interval_states.csv")
HEAT_CSV  = os.path.join(DATA_DIR, "interval_heat.csv")

os.makedirs(DATA_DIR, exist_ok=True)


## CSV Initialization (Append-Safe)

In [ ]:

def init_csv(path, header):
    if not os.path.exists(path):
        with open(path, "w", newline="") as f:
            csv.writer(f).writerow(header)

init_csv(
    STATE_CSV,
    [
        "run_id","step","t",
        "r_lo","r_hi","phi_lo","phi_hi","lam_lo","lam_hi",
        "V_lo","V_hi","gamma_lo","gamma_hi","chi_lo","chi_hi",
        "rho_lo","rho_hi","q_lo","q_hi"
    ]
)

init_csv(
    HEAT_CSV,
    [
        "run_id","step","t",
        "qdot_lo","qdot_hi","Q_lo","Q_hi"
    ]
)


## Initial Conditions and Parameters

In [ ]:

def initial_state():
    return [
        Interval(constants.RADIUS_EARTH + 58_000, constants.RADIUS_EARTH + 62_000), # r: radial distance from Earth center (m)
        Interval(0.30, 0.31),  # phi: geocentric latitude (rad)
        Interval(1.00, 1.01),  # lam: longitude (rad)
        Interval(7600, 7800),  # V: inertial speed (m/s)
        Interval(math.radians(-5.2), math.radians(-4.8)),  # gamma: flight path angle (rad)
        Interval(math.radians(85), math.radians(95)),  # chi: heading angle from north (rad)
    ]

SIGMA_IV = Interval(math.radians(-5), math.radians(5)) 
"""
sigma_iv models bounded bank-angle uncertainty (+/- 5 degrees) to capture guidance dispersions and control errors
It is propagated as an interval through the lift-direction terms, widening the state bounds conservatively over time.
"""

D_m = 5.03 # meters
S_ref = math.pi * (D_m/2)**2  # ~19.9 m^2

VEHICLE = {
    # Mass: NASA quick-facts landed mass as entry proxy
    "mass_kg": 8255.5,                 # approx from 18,200 lb

    # Geometry
    "ref_area_m2": S_ref,              # from 16.5 ft diameter
    "nose_radius_m": 1.0,              # keep as placeholder - NEEDS TO BE CHANGED LATER

    # Aero constants
    "CL_const": 0.40,                  # was 0.3 but changed to 0.4 due to findings from Development of the Orion Crew Module Static Aerodynamic Database, Part I: Hypersonic
    "CD_const": 1.35,                  # was 1.0 originally 
}


## Stop Conditions

In [ ]:
"""
Stop when the trajectory is physically terminated.
The run ends if the entire altitude interval is below ground (impact)
or if the entire speed interval is near zero, indicating the vehicle has effectively stopped.
This ensures only physically meaningful states are propagated in the interval simulation.
"""
def stop_condition(X):
    r, _, _, V, _, _ = X
    h = constants.intv_geometric_altitude(r)
    if h.hi <= 0.0:
        return True
    if V.hi <= 10.0:
        return True
    return False


## Single Interval Run

In [ ]:

def run_single(run_id):
    X = initial_state()
    t = 0.0

    shield = HeatShield(
        radius_m=2.5,
        nose_radius_m=VEHICLE["nose_radius_m"],
        num_rings=5,
    )

    state_rows = []
    heat_rows = []

    for step in range(MAX_STEPS):
        if stop_condition(X):
            break

        try:
            X = interval_euler_step(
                X,
                DT,
                lambda t_, X_: f_interval(t_, X_, VEHICLE, SIGMA_IV),
                t=t
            )
        except ValueError:
            break

        r, phi, lam, V, gamma, chi = X

        z = constants.intv_geometric_altitude(r)
        atm = AtmosphereModel.intv_US_Standard_ATM(z)

        rho_iv = None
        q_iv = None
        for layer in atm.values():
            rho = layer["rho_kgm3"]
            q = 0.5 * rho * V * V
            rho_iv = rho if rho_iv is None else rho_iv.hull(rho)
            q_iv   = q   if q_iv   is None else q_iv.hull(q)

        shield.update(rho_iv, V, DT)

        state_rows.append([
            run_id, step, t,
            r.lo, r.hi, phi.lo, phi.hi, lam.lo, lam.hi,
            V.lo, V.hi, gamma.lo, gamma.hi, chi.lo, chi.hi,
            rho_iv.lo, rho_iv.hi, q_iv.lo, q_iv.hi
        ])

        heat_rows.append([
            run_id, step, t,
            shield.qdot_max().lo, shield.qdot_max().hi,
            shield.Q_max().lo, shield.Q_max().hi
        ])

        t += DT

    return state_rows, heat_rows


## Batch Execution

In [ ]:
all_states = []
all_heat = []

for _ in range(NUM_RUNS):
    run_id = str(uuid.uuid4())

    try:
        s, h = run_single(run_id)

    except (TypeError, ValueError) as e:
        # Interval domain violation (log, division, shape mismatch, etc.)
        print(f"[WARN] Run {run_id} terminated early: {e}")
        continue

    # Only extend if data is well-formed
    if isinstance(s, list) and isinstance(h, list) and len(s) > 0:
        all_states.extend(s)
        all_heat.extend(h)
    else:
        print(f"[WARN] Run {run_id} produced no usable data")

with open(STATE_CSV, "a", newline="") as f:
    csv.writer(f).writerows(all_states)

with open(HEAT_CSV, "a", newline="") as f:
    csv.writer(f).writerows(all_heat)

print(f"Saved {len(all_states)} state rows and {len(all_heat)} heat rows")


## Visualization Utilities

In [ ]:
# Helper function to visualize interval valued quantities.
# The shaded region represents the full interval enclosure [lo, hi],
# while the boundary lines show the lower and upper bounds explicitly.
def plot_interval(t, lo, hi, ax, label):
    ax.fill_between(t, lo, hi, alpha=0.3, label=label)
    ax.plot(t, lo, linewidth=0.8)
    ax.plot(t, hi, linewidth=0.8)


# Plot the interval time history for the final completed trajectory.
# Each subplot shows how uncertainty evolves over time for a key physical quantity.
def plot_last_run(rows):
    # Time history extracted from the stored simulation rows
    t = [r[2] for r in rows]

    # Convenience accessor for a column across all rows
    def col(i):
        return [r[i] for r in rows]

    # Create four vertically stacked plots sharing the same time axis
    fig, axs = plt.subplots(4, 1, figsize=(10, 12), sharex=True)

    # Radial distance interval from Earth center
    plot_interval(t, col(3), col(4), axs[0], "Radius (m)")
    axs[0].set_ylabel("Radius (m)")
    axs[0].grid()

    # Velocity interval showing deceleration during atmospheric entry
    plot_interval(t, col(9), col(10), axs[1], "Velocity (m/s)")
    axs[1].set_ylabel("Velocity (m/s)")
    axs[1].grid()

    # Atmospheric density interval encountered along the trajectory
    # Log scale is used due to the large dynamic range of density
    plot_interval(t, col(15), col(16), axs[2], "Density (kg/m^3)")
    axs[2].set_ylabel("Density (kg/m^3)")
    axs[2].set_yscale("log")
    axs[2].grid()

    # Dynamic pressure interval, a key driver of aerodynamic loads and heating
    # Log scale highlights peak pressure during the entry phase
    plot_interval(t, col(17), col(18), axs[3], "Dynamic Pressure (Pa)")
    axs[3].set_ylabel("Dynamic Pressure (Pa)")
    axs[3].set_xlabel("Time (s)")
    axs[3].set_yscale("log")
    axs[3].grid()

    plt.tight_layout()
    plt.show()


# Select and plot the most recent completed run from the stored simulation data
if all_states:
    last_id = all_states[-1][0]
    plot_last_run([r for r in all_states if r[0] == last_id])

### more visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

R_EARTH = constants.RADIUS_EARTH  # meters


def plot_midpoint_entry_with_3d(rows):
    t_vals = []
    alt_vals = []
    downrange_vals = []
    crossrange_vals = []

    # Reference location
    r0 = rows[0]
    phi0 = 0.5 * (r0[5] + r0[6])
    lam0 = 0.5 * (r0[7] + r0[8])

    for r in rows:
        t = r[2]

        r_mid = 0.5 * (r[3] + r[4])
        phi = 0.5 * (r[5] + r[6])
        lam = 0.5 * (r[7] + r[8])

        # Altitude
        h = r_mid - R_EARTH

        # Central angle
        cos_c = (
            np.sin(phi0) * np.sin(phi)
            + np.cos(phi0) * np.cos(phi) * np.cos(lam - lam0)
        )
        cos_c = np.clip(cos_c, -1.0, 1.0)
        c = np.arccos(cos_c)

        downrange = R_EARTH * c
        crossrange = R_EARTH * np.cos(phi) * (lam - lam0)

        t_vals.append(t)
        alt_vals.append(h / 1e3)
        downrange_vals.append(downrange / 1e3)
        crossrange_vals.append(crossrange / 1e3)

    # 2D plots
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    axs[0].plot(t_vals, alt_vals)
    axs[0].set_xlabel("Time (s)")
    axs[0].set_ylabel("Altitude (km)")
    axs[0].set_title("Altitude vs Time")
    # axs[0].invert_yaxis()
    axs[0].grid(True)

    axs[1].plot(downrange_vals, alt_vals)
    axs[1].set_xlabel("Downrange (km)")
    axs[1].set_ylabel("Altitude (km)")
    axs[1].set_title("Entry Profile")
    # axs[1].invert_yaxis()
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()

    # 3D plots
    fig = plt.figure(figsize=(14, 5))

    # 3D spatial corridor
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.plot(downrange_vals, crossrange_vals, alt_vals)
    ax1.set_xlabel("Downrange (km)")
    ax1.set_ylabel("Crossrange (km)")
    ax1.set_zlabel("Altitude (km)")
    ax1.set_title("3D Entry Corridor")
    # ax1.invert_zaxis()
    ax1.grid(True)

    # 3D space-time descent
    ax2 = fig.add_subplot(122, projection="3d")
    ax2.plot(t_vals, downrange_vals, alt_vals)
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Downrange (km)")
    ax2.set_zlabel("Altitude (km)")
    ax2.set_title("3D Space-Time Descent")
    # ax2.invert_zaxis()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


if all_states:
    last_id = all_states[-1][0]
    plot_midpoint_entry_with_3d([r for r in all_states if r[0] == last_id])


In [ ]:
import os, csv
import numpy as np
import matplotlib.pyplot as plt
import constants

def get_last_run_rows():
    """
    Returns the rows for the most recent run_id.
    Works with in-memory all_states OR falls back to reading STATE_CSV.
    Row layout matches notebook header:
    [run_id, step, t,
     r_lo, r_hi, phi_lo, phi_hi, lam_lo, lam_hi,
     V_lo, V_hi, gamma_lo, gamma_hi, chi_lo, chi_hi,
     rho_lo, rho_hi, q_lo, q_hi]
    """
    # 1) In-memory path (normal notebook flow)
    if "all_states" in globals() and all_states:
        last_id = all_states[-1][0]
        return [r for r in all_states if r[0] == last_id]

    # 2) Fallback to reading the CSV if kernel was restarted
    if "STATE_CSV" in globals() and os.path.exists(STATE_CSV):
        rows = []
        with open(STATE_CSV, "r", newline="") as f:
            reader = csv.reader(f)
            header = next(reader, None)
            for row in reader:
                # parse types
                run_id = row[0]
                step = int(row[1])
                t = float(row[2])
                rest = [float(x) for x in row[3:]]
                rows.append([run_id, step, t] + rest)

        if not rows:
            raise RuntimeError("STATE_CSV exists but contains no data rows.")

        last_id = rows[-1][0]
        return [r for r in rows if r[0] == last_id]

    raise RuntimeError("No in-memory all_states and no STATE_CSV found. Run the sim cells first.")


rows = get_last_run_rows()
print("Loaded rows:", len(rows), " last run_id:", rows[0][0])


In [ ]:
rows = get_last_run_rows()

t = np.array([r[2] for r in rows])

# Column indices 
r_lo = np.array([r[3]  for r in rows]); r_hi = np.array([r[4]  for r in rows])
V_lo = np.array([r[9]  for r in rows]); V_hi = np.array([r[10] for r in rows])
rho_lo = np.array([r[15] for r in rows]); rho_hi = np.array([r[16] for r in rows])
q_lo = np.array([r[17] for r in rows]); q_hi = np.array([r[18] for r in rows])

r_w   = r_hi - r_lo
V_w   = V_hi - V_lo
rho_w = rho_hi - rho_lo
q_w   = q_hi - q_lo

fig, axs = plt.subplots(2, 2, figsize=(12, 7), sharex=True)

axs[0,0].plot(t, r_w)
axs[0,0].set_ylabel("Radius width (m)")
axs[0,0].set_title("Uncertainty growth: Radius")
axs[0,0].grid(True)

axs[0,1].plot(t, V_w)
axs[0,1].set_ylabel("Velocity width (m/s)")
axs[0,1].set_title("Uncertainty growth: Velocity")
axs[0,1].grid(True)

axs[1,0].plot(t, rho_w)
axs[1,0].set_ylabel("Density width (kg/m³)")
axs[1,0].set_title("Uncertainty growth: Density")
axs[1,0].set_yscale("log")
axs[1,0].grid(True)

axs[1,1].plot(t, q_w)
axs[1,1].set_ylabel("q width (Pa)")
axs[1,1].set_title("Uncertainty growth: Dynamic Pressure")
axs[1,1].set_yscale("log")
axs[1,1].grid(True)

for ax in axs[1,:]:
    ax.set_xlabel("Time (s)")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rows = get_last_run_rows()
t = np.array([r[2] for r in rows])

def asymmetry(lo, hi):
    mid = 0.5*(lo + hi)
    return ((hi - mid) - (mid - lo)) / (hi - lo + 1e-12)

V_lo = np.array([r[9]  for r in rows])
V_hi = np.array([r[10] for r in rows])
q_lo = np.array([r[17] for r in rows])
q_hi = np.array([r[18] for r in rows])

plt.figure(figsize=(8,5))
plt.plot(t, asymmetry(V_lo, V_hi), label="Velocity")
plt.plot(t, asymmetry(q_lo, q_hi), label="Dynamic pressure")
plt.axhline(0, color="k", linewidth=0.8)
plt.xlabel("Time (s)")
plt.ylabel("Asymmetry index")
plt.title("Directional asymmetry in interval growth")
plt.legend()
plt.grid(True)
plt.show()


* **Velocity (blue):** its uncertainty grows unevenly at first, meaning one side of the interval runs away faster than the other.
* **Dynamic pressure (orange):** its uncertainty grows evenly on both sides, staying balanced over time.


### interval growth rate

In [ ]:
r_lo = np.array([r[3] for r in rows])
r_hi = np.array([r[4] for r in rows])

r_width = r_hi - r_lo
drw_dt = np.gradient(r_width, t)

plt.figure(figsize=(8,5))
plt.plot(t, drw_dt)
plt.xlabel("Time (s)")
plt.ylabel("d(width)/dt  [m/s]")
plt.title("Rate of interval growth (radius)")
plt.grid(True)
plt.show()


### when does uncertainty become physically dominant

In [ ]:
def normalized_width(lo, hi):
    mid = 0.5*(lo + hi)
    return (hi - lo) / (np.abs(mid) + 1e-12)

rho_lo = np.array([r[15] for r in rows])
rho_hi = np.array([r[16] for r in rows])

plt.figure(figsize=(8,5))
plt.plot(t, normalized_width(rho_lo, rho_hi))
plt.yscale("log")
plt.xlabel("Time (s)")
plt.ylabel("Relative uncertainty")
plt.title("Normalized density uncertainty growth")
plt.grid(True)
plt.show()


### Compute correlations between interval widths:

Does velocity uncertainty drive q uncertainty?

Does radius uncertainty drive density uncertainty?

In [ ]:
import pandas as pd
import seaborn as sns

data = {
    "r_width": r_hi - r_lo,
    "V_width": V_hi - V_lo,
    "rho_width": rho_hi - rho_lo,
    "q_width": q_hi - q_lo,
}

df = pd.DataFrame(data)
corr = df.corr()

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation of interval width growth")
plt.show()


### Peak Heat Flux Interval vs Time

In [ ]:
# ---------------- Peak Heat Flux Interval vs Time ----------------
# Load heating data from interval_heat.csv.

import csv
import matplotlib.pyplot as plt

def load_interval_heat(path="data_intv/interval_heat.csv"):
    rows = []
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        for r in reader:
            run_id = r[0]                 # keep UUID as string
            step = int(r[1])
            time = float(r[2])
            qdot_lo = float(r[3])
            qdot_hi = float(r[4])
            Q_lo = float(r[5])
            Q_hi = float(r[6])

            rows.append([
                run_id,
                step,
                time,
                qdot_lo,
                qdot_hi,
                Q_lo,
                Q_hi,
            ])

    return header, rows


def plot_peak_heating_from_heat_csv(rows):
    # rows layout:
    # [run_id, step, time, qdot_lo, qdot_hi, Q_lo, Q_hi]

    t = [r[2] for r in rows]
    qdot_lo = [r[3] for r in rows]
    qdot_hi = [r[4] for r in rows]

    fig, ax = plt.subplots(1, 1, figsize=(10, 4))

    ax.fill_between(t, qdot_lo, qdot_hi, alpha=0.3, label="Peak heat flux interval")
    ax.plot(t, qdot_lo, linewidth=0.8)
    ax.plot(t, qdot_hi, linewidth=0.8)

    ax.set_ylabel("Heat Flux qdot (W/m^2)")
    ax.set_xlabel("Time (s)")
    ax.set_yscale("log")
    ax.grid()
    ax.legend()

    plt.tight_layout()
    plt.show()


# Load heating data and plot only the most recent run
heat_header, heat_rows = load_interval_heat()

if heat_rows:
    last_run_id = heat_rows[-1][0]
    last_rows = [r for r in heat_rows if r[0] == last_run_id]
    plot_peak_heating_from_heat_csv(last_rows)


### 

### Integrated Heat Load Interval vs Altitude

In [ ]:
# ---------------- Integrated Heat Load Interval vs Altitude ----------------
# Heat is loaded from interval_heat.csv
# Altitude is reconstructed from interval_states.csv using radius midpoints

import csv
import matplotlib.pyplot as plt

def load_interval_states(path="data_intv/interval_states.csv"):
    rows = []
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        for r in reader:
            run_id = r[0]                 # UUID string
            step = int(r[1])
            r_lo = float(r[3])
            r_hi = float(r[4])
            r_mid = 0.5 * (r_lo + r_hi)

            rows.append([
                run_id,
                step,
                r_mid,
            ])

    return header, rows


def load_interval_heat(path="data_intv/interval_heat.csv"):
    rows = []
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        for r in reader:
            run_id = r[0]                 # UUID string
            step = int(r[1])
            time = float(r[2])
            qdot_lo = float(r[3])
            qdot_hi = float(r[4])
            Q_lo = float(r[5])
            Q_hi = float(r[6])

            rows.append([
                run_id,
                step,
                time,
                qdot_lo,
                qdot_hi,
                Q_lo,
                Q_hi,
            ])

    return header, rows


def plot_heat_load_vs_altitude():
    _, state_rows = load_interval_states()
    _, heat_rows = load_interval_heat()

    # Build lookup table for radius midpoint
    state_lookup = {}
    for r in state_rows:
        state_lookup[(r[0], r[1])] = r[2]

    # Use only most recent run
    last_run_id = heat_rows[-1][0]
    rows = [r for r in heat_rows if r[0] == last_run_id]

    h_km = []
    Q_lo = []
    Q_hi = []

    for r in rows:
        key = (r[0], r[1])
        if key not in state_lookup:
            continue

        r_mid = state_lookup[key]
        h_mid = (r_mid - constants.RADIUS_EARTH) / 1000.0

        h_km.append(h_mid)
        Q_lo.append(r[5])
        Q_hi.append(r[6])

    fig, ax = plt.subplots(1, 1, figsize=(10, 4))

    ax.fill_between(h_km, Q_lo, Q_hi, alpha=0.3, label="Integrated heat load interval")
    ax.plot(h_km, Q_lo, linewidth=0.8)
    ax.plot(h_km, Q_hi, linewidth=0.8)

    ax.set_xlabel("Altitude (km)")
    ax.set_ylabel("Integrated Heat Load Q (J/m^2)")
    ax.invert_xaxis()
    ax.grid()
    ax.legend()

    plt.tight_layout()
    plt.show()


plot_heat_load_vs_altitude()
